In [1]:
import numpy as np
import math
import pandas as pd

# 1. Definimos 3 clases representativas de tu dataset
clases = ['ACTIVIDAD NO ESTRUCTURADA', 'CAMINAR CON LA COMPRA', 'SIT TO STAND 30 s']

# 2. Simulamos la cantidad de datos ETIQUETADOS (Ground Truth) que le pasas al modelo.
# Asumimos que tienes solo una pequeña fracción etiquetada para el Self-Training.
etiquetas_reales = {
    'ACTIVIDAD NO ESTRUCTURADA': 50000, 
    'CAMINAR CON LA COMPRA': 20, 
    'SIT TO STAND 30 s': 5
}

# 3. Inicializamos el contador de pseudo-etiquetas (las que el modelo irá prediciendo).
# Al principio, el modelo no ha predicho nada.
pseudo_etiquetas_validas = {
    'ACTIVIDAD NO ESTRUCTURADA': 0, 
    'CAMINAR CON LA COMPRA': 0, 
    'SIT TO STAND 30 s': 0
}

print("Datos etiquetados iniciales (Ground Truth):", etiquetas_reales)

Datos etiquetados iniciales (Ground Truth): {'ACTIVIDAD NO ESTRUCTURADA': 50000, 'CAMINAR CON LA COMPRA': 20, 'SIT TO STAND 30 s': 5}


In [2]:
def recalcular_umbrales_sdt(etiquetas_reales, pseudo_etiquetas, lambda_param=10, gamma_base=0.95):
    """
    Calcula los umbrales dinámicos adaptativos según el paper FldtMatch.
    """
    # PASO 1: CEL (Cumulative Effective Labeling)
    # Sumamos las etiquetas reales + las pseudo-etiquetas que hemos dado por buenas
    alpha = {}
    for c in clases:
        alpha[c] = etiquetas_reales[c] + pseudo_etiquetas[c]
        
    # Encontramos la clase más representada (la que tiene el alpha más alto)
    max_alpha = max(alpha.values())
    
    umbrales = {}
    dificultades = {}
    
    # PASO 2: SDT (Self-adaptive Dynamic Threshold)
    for c in clases:
        # Calculamos la dificultad (Sigma): va de ~0 (muy difícil/rara) a 1 (muy fácil/común)
        sigma = alpha[c] / max_alpha
        dificultades[c] = sigma
        
        # Aplicamos la función logarítmica protectora del paper
        # M(x) = log_lambda(sigma + lambda - 1)
        multiplicador = math.log(sigma + lambda_param - 1, lambda_param)
        
        # Obtenemos el umbral final
        umbral_final = multiplicador * gamma_base
        umbrales[c] = round(umbral_final, 4)
        
    return umbrales, dificultades, alpha

# Probamos la función en el "Paso 0" (antes de entrenar)
umbrales_iniciales, _, _ = recalcular_umbrales_sdt(etiquetas_reales, pseudo_etiquetas_validas)
print("Umbrales antes de empezar a predecir:", umbrales_iniciales)

Umbrales antes de empezar a predecir: {'ACTIVIDAD NO ESTRUCTURADA': 0.95, 'CAMINAR CON LA COMPRA': 0.9065, 'SIT TO STAND 30 s': 0.9065}


In [3]:
# Parámetros del SDT
LAMBDA = 10
UMBRAL_FIJO = 0.95
EPOCAS = 5

historial = []

print("INICIANDO ENTRENAMIENTO...\n")

for epoca in range(1, EPOCAS + 1):
    # 1. Al inicio de la época, calculamos los umbrales dinámicos actuales
    umbrales, dificultades, alphas = recalcular_umbrales_sdt(
        etiquetas_reales, 
        pseudo_etiquetas_validas, 
        lambda_param=LAMBDA, 
        gamma_base=UMBRAL_FIJO
    )
    
    # Guardamos los datos para verlos luego en una tabla
    historial.append({
        'Época': epoca,
        'Umbral NO ESTRUCTURADA': umbrales['ACTIVIDAD NO ESTRUCTURADA'],
        'Umbral CAMINAR': umbrales['CAMINAR CON LA COMPRA'],
        'Umbral SIT TO STAND': umbrales['SIT TO STAND 30 s']
    })
    
    # 2. SIMULACIÓN DE PREDICCIONES: 
    # El modelo pasa por las imágenes sin etiquetar.
    # Simulamos que encuentra muchas más predicciones de la clase mayoritaria que superan el umbral
    pseudo_etiquetas_validas['ACTIVIDAD NO ESTRUCTURADA'] += 15000 
    
    # Como los umbrales de las clases raras son más bajos y permisivos, 
    # el modelo logra captar algunas pseudo-etiquetas válidas de estas clases.
    pseudo_etiquetas_validas['CAMINAR CON LA COMPRA'] += 20
    pseudo_etiquetas_validas['SIT TO STAND 30 s'] += 3
    
    # (En tu código real, esto ocurre comprobando si max(prediccion) > umbrales[clase_predicha])

# Mostramos cómo han evolucionado los umbrales
df_historial = pd.DataFrame(historial)
print("Evolución de los umbrales por época:")
display(df_historial) # Usa print(df_historial) si no estás en Jupyter

print("\nPseudo-etiquetas finales acumuladas:", pseudo_etiquetas_validas)

INICIANDO ENTRENAMIENTO...

Evolución de los umbrales por época:


,Época,Umbral NO ESTRUCTURADA,Umbral CAMINAR,Umbral SIT TO STAND
0,1,0.95,0.9065,0.9065
1,2,0.95,0.9066,0.9065
2,3,0.95,0.9066,0.9065
3,4,0.95,0.9066,0.9065
4,5,0.95,0.9066,0.9065



Pseudo-etiquetas finales acumuladas: {'ACTIVIDAD NO ESTRUCTURADA': 75000, 'CAMINAR CON LA COMPRA': 100, 'SIT TO STAND 30 s': 15}
